# Train Test Creator

## Install libraries

In [109]:
import os
import sys
import random
from dotenv import load_dotenv
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from utils.utils import *
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from dtos.tabular_database_driver_dtos.tabular_database_driver_dtos import *
from ta.ta_functions import *

load_dotenv()

True

In [110]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Parameters

In [111]:
STOCK_CODE = "VCB"
LOOKBACK_WINDOW = 20
FORECAST_HORIZON = 5
STRIDE = 5
ID_COLUMN = ["date", "code"]
TARGET_COLUMN = f"adjust"

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2022-12-31")
VAL_RANGE = ("2023-01-01", "2024-06-30")
TEST_RANGE = ("2024-07-01", "2026-04-30")

In [112]:
STOCK_CODE = str.lower(STOCK_CODE)
STOCK_CODE

'vcb'

In [113]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

25

## Load data

In [114]:
my_logger = Logger(
    file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/{STOCK_CODE}/train_test_creator.log",
)

In [115]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [116]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [117]:
stock_df = my_postgresql_driver.select(
    schema_name=Schema.ENTERPRISE.value,
    table_name=f"unified_{STOCK_CODE}",
    order_by=["date"],
)

# cast all string columns that look numeric → float
for col in stock_df.columns:
    if stock_df[col].dtype == object:
        converted = pd.to_numeric(stock_df[col], errors="coerce")
        if converted.notna().sum() / len(stock_df) >= 0.9:
            stock_df[col] = converted

stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,upcom_index_date_month_cos,upcom_index_date_dow_sin,upcom_index_date_dow_cos,upcom_index_date_hour_sin,upcom_index_date_hour_cos,upcom_index_date_quarter_sin,upcom_index_date_quarter_cos,upcom_index_date_doy_sin,upcom_index_date_doy_cos,upcom_index_date_unix_ts
0,VCB,2009-06-30,60.0,9.13,10.0,294070,17.64,60.0,60.0,60.0,...,-1.000000,0.781831,0.623490,0.0,1.0,1.224647e-16,-1.000000e+00,0.025818,-0.999667,1.246320e+09
1,VCB,2009-07-01,60.5,9.21,0.5,6248390,389.79,63.0,63.0,59.5,...,-0.866025,0.974928,-0.222521,0.0,1.0,-1.000000e+00,-1.836970e-16,0.008607,-0.999963,1.246406e+09
2,VCB,2009-07-02,58.0,8.83,-2.5,1515670,88.93,59.5,60.0,57.5,...,-0.866025,0.433884,-0.900969,0.0,1.0,-1.000000e+00,-1.836970e-16,-0.008607,-0.999963,1.246493e+09
3,VCB,2009-07-03,56.0,8.53,-2.0,899720,50.68,56.5,57.0,56.0,...,-0.866025,-0.433884,-0.900969,0.0,1.0,-1.000000e+00,-1.836970e-16,-0.025818,-0.999667,1.246579e+09
4,VCB,2009-07-06,58.5,8.91,2.5,1571740,90.18,56.0,58.5,56.0,...,-0.866025,0.000000,1.000000,0.0,1.0,-1.000000e+00,-1.836970e-16,-0.077386,-0.997001,1.246838e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,-0.500000,0.974928,-0.222521,0.0,1.0,1.224647e-16,-1.000000e+00,0.936881,-0.349647,1.776816e+09
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,-0.500000,0.433884,-0.900969,0.0,1.0,1.224647e-16,-1.000000e+00,0.930724,-0.365723,1.776902e+09
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,-0.500000,-0.433884,-0.900969,0.0,1.0,1.224647e-16,-1.000000e+00,0.924291,-0.381689,1.776989e+09
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,-0.500000,0.781831,0.623490,0.0,1.0,1.224647e-16,-1.000000e+00,0.895839,-0.444378,1.777334e+09


## Create features

In [118]:
N_LIST = [5]

In [119]:
feature_functions = [
    lambda df: add_bbands(
        df,
        n=N_LIST,
    ),
    lambda df: add_dema(
        df,
        n=N_LIST,
    ),
    lambda df: add_ema(
        df,
        n=N_LIST,
    ),
    lambda df: add_kama(
        df,
        n=N_LIST,
    ),
    lambda df: add_midpoint(
        df,
        n=N_LIST,
    ),
    lambda df: add_midprice(
        df,
        n=N_LIST,
    ),
    lambda df: add_sar(df),
    lambda df: add_sma(
        df,
        n=N_LIST,
    ),
    lambda df: add_t3(
        df,
        n=N_LIST,
    ),
    lambda df: add_tema(
        df,
        n=N_LIST,
    ),
    lambda df: add_trima(
        df,
        n=N_LIST,
    ),
    lambda df: add_wma(
        df,
        n=N_LIST,
    ),
    lambda df: add_adx(
        df,
        n=N_LIST,
    ),
    lambda df: add_aroon(
        df,
        n=N_LIST,
    ),
    lambda df: add_bop(
        df,
        n=N_LIST,
    ),
    lambda df: add_cci(
        df,
        n=N_LIST,
    ),
    lambda df: add_cmo(
        df,
        n=N_LIST,
    ),
    lambda df: add_macd(
        df,
    ),
    lambda df: add_mfi(
        df,
        n=N_LIST,
    ),
    lambda df: add_mom(
        df,
        n=N_LIST,
    ),
    lambda df: add_ppo(
        df,
    ),
    lambda df: add_roc(
        df,
        n=N_LIST,
    ),
    lambda df: add_rsi(
        df,
        n=N_LIST,
    ),
    lambda df: add_stoch(
        df,
    ),
    lambda df: add_stoch_rsi(
        df,
        n=N_LIST,
    ),
    lambda df: add_trix(
        df,
        n=N_LIST,
    ),
    lambda df: add_ultosc(
        df,
    ),
    lambda df: add_willr(
        df,
        n=N_LIST,
    ),
    lambda df: add_ad(
        df,
        n=N_LIST,
    ),
    lambda df: add_adosc(
        df,
    ),
    lambda df: add_obv(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_dcperiod(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_dcphase(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_phasor(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_sine(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_trendmode(
        df,
        n=N_LIST,
    ),
]
len(feature_functions)

36

In [120]:
def apply_features(df, funcs):
    for func in funcs:
        df = func(df)
    return df


featured_stock_df = apply_features(stock_df, feature_functions)
featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
0,VCB,2009-06-30,60.0,9.13,10.0,294070,17.64,60.0,60.0,60.0,...,True,0.000000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
1,VCB,2009-07-01,60.5,9.21,0.5,6248390,389.79,63.0,63.0,59.5,...,True,0.000000,0.000000,0.000000,0.000000,NaN,False,False,0.000000,0.000000
2,VCB,2009-07-02,58.0,8.83,-2.5,1515670,88.93,59.5,60.0,57.5,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
3,VCB,2009-07-03,56.0,8.53,-2.0,899720,50.68,56.5,57.0,56.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
4,VCB,2009-07-06,58.5,8.91,2.5,1571740,90.18,56.0,58.5,56.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,True,0.058528,-0.029264,-0.058528,0.029264,-0.014632,False,True,0.058528,0.000000
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,True,0.372352,0.313824,0.627648,0.686176,0.656912,True,False,0.627648,0.627648
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,True,0.581568,0.209216,0.418432,-0.209216,-0.895392,True,False,0.418432,0.000000
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,True,0.721045,0.139477,0.278955,-0.139477,0.069739,True,False,0.278955,0.000000


In [121]:
# Drop rows with missing values
featured_stock_df = featured_stock_df.ffill().dropna()
featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
65,VCB,2009-09-30,53.5,8.14,-0.5,712000,38.11,54.0,54.5,53.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
66,VCB,2009-10-01,52.5,7.99,-1.0,875180,46.24,53.5,53.5,52.5,...,True,0.333333,0.333333,0.666667,0.666667,0.666667,True,False,0.666667,0.666667
67,VCB,2009-10-02,51.0,7.76,-1.5,1416100,72.10,51.5,51.5,50.0,...,True,0.555556,0.222222,0.444444,-0.222222,-0.888889,True,False,0.444444,0.000000
68,VCB,2009-10-05,51.0,7.76,0.0,606370,30.78,51.5,51.5,50.0,...,True,0.703704,0.148148,0.296296,-0.148148,0.074074,True,False,0.296296,0.000000
69,VCB,2009-10-06,50.5,7.69,-0.5,460990,23.38,51.5,51.5,50.5,...,True,0.802469,0.098765,0.197531,-0.098765,0.049383,True,False,0.197531,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,True,0.058528,-0.029264,-0.058528,0.029264,-0.014632,False,True,0.058528,0.000000
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,True,0.372352,0.313824,0.627648,0.686176,0.656912,True,False,0.627648,0.627648
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,True,0.581568,0.209216,0.418432,-0.209216,-0.895392,True,False,0.418432,0.000000
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,True,0.721045,0.139477,0.278955,-0.139477,0.069739,True,False,0.278955,0.000000


## Split Train Val Test

In [122]:
train_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TRAIN_RANGE[0])
    & (featured_stock_df["date"] <= TRAIN_RANGE[1])
]
train_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
65,VCB,2009-09-30,53.5,8.14,-0.5,712000,38.11,54.0,54.5,53.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
66,VCB,2009-10-01,52.5,7.99,-1.0,875180,46.24,53.5,53.5,52.5,...,True,0.333333,0.333333,0.666667,0.666667,0.666667,True,False,0.666667,0.666667
67,VCB,2009-10-02,51.0,7.76,-1.5,1416100,72.10,51.5,51.5,50.0,...,True,0.555556,0.222222,0.444444,-0.222222,-0.888889,True,False,0.444444,0.000000
68,VCB,2009-10-05,51.0,7.76,0.0,606370,30.78,51.5,51.5,50.0,...,True,0.703704,0.148148,0.296296,-0.148148,0.074074,True,False,0.296296,0.000000
69,VCB,2009-10-06,50.5,7.69,-0.5,460990,23.38,51.5,51.5,50.5,...,True,0.802469,0.098765,0.197531,-0.098765,0.049383,True,False,0.197531,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3314,VCB,2022-12-26,78.9,44.36,-0.2,1203600,96.24,79.2,80.8,78.9,...,True,0.007607,-0.003804,-0.007607,0.003804,-0.001902,False,True,0.007607,0.000000
3315,VCB,2022-12-27,79.1,44.48,0.2,1058000,84.38,80.2,80.5,78.9,...,True,0.005071,-0.002536,-0.005071,0.002536,-0.001268,False,True,0.005071,0.000000
3316,VCB,2022-12-28,80.0,44.98,0.9,1173500,94.81,80.4,82.0,79.5,...,True,0.003381,-0.001690,-0.003381,0.001690,-0.000845,False,True,0.003381,0.000000
3317,VCB,2022-12-29,80.7,45.38,0.7,1237400,100.75,82.3,82.5,80.5,...,True,0.002254,-0.001127,-0.002254,0.001127,-0.000563,False,True,0.002254,0.000000


In [123]:
val_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= VAL_RANGE[0])
    & (featured_stock_df["date"] <= VAL_RANGE[1])
]
val_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
3319,VCB,2023-01-03,82.6,46.44,2.6,1089800,88.38,80.2,82.6,80.2,...,True,0.556557,0.221721,0.443443,-0.221721,-0.889139,True,False,0.443443,0.0
3320,VCB,2023-01-04,82.8,46.56,0.2,837900,69.17,82.9,82.9,81.8,...,True,0.704372,0.147814,0.295628,-0.147814,0.073907,True,False,0.295628,0.0
3321,VCB,2023-01-05,84.0,47.23,1.2,1078600,89.68,82.8,84.0,82.5,...,True,0.802914,0.098543,0.197086,-0.098543,0.049271,True,False,0.197086,0.0
3322,VCB,2023-01-06,84.0,47.23,0.0,1125500,94.85,84.0,84.8,83.9,...,True,0.868610,0.065695,0.131390,-0.065695,0.032848,True,False,0.131390,0.0
3323,VCB,2023-01-09,86.9,48.86,2.9,1667100,142.86,85.8,86.9,84.8,...,True,0.912406,0.043797,0.087594,-0.043797,0.021898,True,False,0.087594,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3682,VCB,2024-06-24,85.5,56.78,-0.5,2149800,185.89,86.0,87.4,85.5,...,True,0.999324,0.000338,0.000676,-0.000338,0.000169,True,False,0.000676,0.0
3683,VCB,2024-06-25,85.5,56.78,0.0,1565900,135.09,85.7,86.7,85.5,...,True,0.999549,0.000225,0.000451,-0.000225,0.000113,True,False,0.000451,0.0
3684,VCB,2024-06-26,85.2,56.58,-0.3,1706900,146.32,85.9,86.3,85.2,...,True,0.999700,0.000150,0.000300,-0.000150,0.000075,True,False,0.000300,0.0
3685,VCB,2024-06-27,85.2,56.58,0.0,1423000,121.56,85.2,86.0,85.2,...,True,0.999800,0.000100,0.000200,-0.000100,0.000050,True,False,0.000200,0.0


In [124]:
test_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TEST_RANGE[0])
    & (featured_stock_df["date"] <= TEST_RANGE[1])
]
test_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
3687,VCB,2024-07-01,86.1,57.18,0.9,1179200,100.98,85.8,86.2,85.2,...,True,0.999911,0.000044,0.000089,-0.000044,0.000022,True,False,0.000089,0.000000
3688,VCB,2024-07-02,88.2,58.57,2.1,2425200,213.04,86.7,88.8,86.2,...,True,0.666607,-0.333304,-0.666607,-0.666696,-0.666652,False,True,0.666607,0.666607
3689,VCB,2024-07-03,88.5,58.77,0.3,2250700,198.80,88.0,88.7,87.8,...,True,0.777738,0.111131,0.222262,0.888869,1.555565,True,False,0.222262,0.222262
3690,VCB,2024-07-04,88.0,58.44,-0.5,3060300,271.90,88.6,89.7,88.0,...,True,0.518492,-0.259246,-0.518492,-0.740754,-1.629623,False,True,0.518492,0.518492
3691,VCB,2024-07-05,88.0,58.44,0.0,1847400,163.71,88.1,89.2,88.0,...,True,0.345661,-0.172831,-0.345661,0.172831,0.913585,False,True,0.345661,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,True,0.058528,-0.029264,-0.058528,0.029264,-0.014632,False,True,0.058528,0.000000
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,True,0.372352,0.313824,0.627648,0.686176,0.656912,True,False,0.627648,0.627648
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,True,0.581568,0.209216,0.418432,-0.209216,-0.895392,True,False,0.418432,0.000000
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,True,0.721045,0.139477,0.278955,-0.139477,0.069739,True,False,0.278955,0.000000


## Standardization

In [125]:
ordinal_map = {"ht_dcphase_quadrant": [1, 2, 3, 4]}

In [126]:
def categorize_columns(df, ordinal_map: dict = None):
    """
    Auto-cast columns to suitable dtypes, then categorize into 3 lists.

    Parameters
    ----------
    df : pd.DataFrame
    ordinal_map : dict, optional
        {col_name: [ordered_categories]} for columns that should be ordinal.
        Example: {"size": ["S", "M", "L"], "priority": ["low", "med", "high"]}

    Returns
    -------
    numerical, nominal_categorical, ordinal_categorical : list of column names
    """
    ordinal_map = ordinal_map or {}
    df = df.copy()

    for col in df.columns:
        # --- 1. Try casting object/string columns ---
        if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            # Try numeric first
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() / len(df) >= 0.9:  # 90%+ parseable → numeric
                df[col] = converted
            else:
                # Fall through to categorical casting below
                pass

        # --- 2. Cast to ordinal categorical ---
        if col in ordinal_map:
            df[col] = pd.Categorical(df[col], categories=ordinal_map[col], ordered=True)

        # --- 3. Cast remaining object/string → nominal categorical ---
        elif df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            df[col] = pd.Categorical(df[col])

    # --- 4. Categorize ---
    numerical, nominal_categorical, ordinal_categorical = [], [], []

    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_numeric_dtype(dtype):
            numerical.append(col)
        elif isinstance(dtype, pd.CategoricalDtype):
            if dtype.ordered:
                ordinal_categorical.append(col)
            else:
                nominal_categorical.append(col)

    return numerical, nominal_categorical, ordinal_categorical

In [127]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)
numerical, nominal, ordinal

(['close',
  'adjust',
  'change',
  'matching_volume',
  'matching_value',
  'open',
  'high',
  'low',
  'percent_change',
  'number_of_buy_orders',
  'buy_volume',
  'average_volume_per_buy_order',
  'number_of_sell_orders',
  'sell_volume',
  'average_volume_per_sell_order',
  'net_volume',
  'date_year',
  'date_month',
  'date_day',
  'date_week',
  'date_day_of_week',
  'date_day_of_year',
  'date_quarter',
  'date_day_of_quarter',
  'date_days_to_quarter_end',
  'date_quarter_progress',
  'date_days_in_month',
  'date_days_to_month_end',
  'date_week_of_month',
  'date_days_in_year',
  'date_days_to_year_end',
  'date_year_progress',
  'date_is_leap_year',
  'date_is_month_start',
  'date_is_month_end',
  'date_is_quarter_start',
  'date_is_quarter_end',
  'date_is_year_end',
  'date_month_sin',
  'date_month_cos',
  'date_dow_sin',
  'date_dow_cos',
  'date_quarter_sin',
  'date_quarter_cos',
  'date_doy_sin',
  'date_doy_cos',
  'date_unix_ts',
  'vnindex_open',
  'vnindex_hi

In [128]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)

numerical = [c for c in numerical if c not in ID_COLUMN and c != TARGET_COLUMN]

TIME_TREND_COLS = [
    "date_year", "date_unix_ts",
    "vnindex_date_year", "vnindex_date_unix_ts",
    "hnx_index_date_year", "hnx_index_date_unix_ts",
    "upcom_index_date_year", "upcom_index_date_unix_ts",
]
numerical = [c for c in numerical if c not in TIME_TREND_COLS]

nominal = [c for c in nominal if c not in ID_COLUMN and c != TARGET_COLUMN]
ordinal = [c for c in ordinal if c not in ID_COLUMN and c != TARGET_COLUMN]

ordinal_categories = [ordinal_map[col] for col in ordinal]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal),
        ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal),
    ],
    remainder="drop",
)

train_featured_scaled_stock_tensor = preprocessor.fit_transform(
    train_featured_stock_df
)  # fit+transform
val_featured_scaled_stock_tensor = preprocessor.transform(
    val_featured_stock_df
)  # transform only
test_featured_scaled_stock_tensor = preprocessor.transform(
    test_featured_stock_df
)  # transform only

In [129]:
display(train_featured_scaled_stock_tensor)
train_featured_scaled_stock_tensor.shape

array([[ 0.08542267, -0.49489787, -0.28775842, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.04381514, -0.96148234, -0.08519269, ...,  1.        ,
         0.        ,  1.        ],
       [-0.01859616, -1.42806682,  0.58628574, ...,  1.        ,
         0.        ,  2.        ],
       ...,
       [ 1.18802229,  0.81153867,  0.28513091, ...,  1.        ,
         0.        ,  2.        ],
       [ 1.21714757,  0.62490488,  0.36445405, ...,  1.        ,
         0.        ,  2.        ],
       [ 1.18802229, -0.68153166,  0.70396701, ...,  1.        ,
         0.        ,  2.        ]], shape=(3254, 781))

(3254, 781)

In [130]:
display(val_featured_scaled_stock_tensor)
val_featured_scaled_stock_tensor.shape

array([[ 1.29620188,  2.39792589,  0.18122877, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.30452339,  0.1583204 , -0.13147073, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.35445243,  1.09148935,  0.1673255 , ...,  1.        ,
         0.        ,  1.        ],
       ...,
       [ 1.40438146, -0.30826408,  0.94727429, ...,  1.        ,
         0.        ,  3.        ],
       [ 1.40438146, -0.02831339,  0.59485115, ...,  1.        ,
         0.        ,  3.        ],
       [ 1.40438146, -0.02831339,  0.20928359, ...,  1.        ,
         0.        ,  3.        ]], shape=(368, 781))

(368, 781)

In [131]:
display(test_featured_scaled_stock_tensor)
test_featured_scaled_stock_tensor.shape

array([[ 1.44182824,  0.81153867,  0.29220669, ...,  1.        ,
         0.        ,  3.        ],
       [ 1.52920406,  1.93134141,  1.8389458 , ...,  1.        ,
         0.        ,  0.        ],
       [ 1.54168632,  0.2516373 ,  1.62232785, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [ 0.38083615, -2.08128509, 13.57889433, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.34755013, -0.77484855,  8.25828555, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.34755013, -0.02831339,  6.81160035, ...,  1.        ,
         0.        ,  2.        ]], shape=(444, 781))

(444, 781)

## Roll windows

In [132]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

25

In [133]:
def make_windows(
    X_scaled, source_df, target_col, lookback_window, forecast_horizon, stride=1
):
    prices = source_df[target_col].values
    dates = source_df["date"].values

    total_window = lookback_window + forecast_horizon
    X_list, y_list, dates_list = [], [], []

    for i in range(0, len(X_scaled) - total_window + 1, stride):
        today_idx = i + lookback_window - 1
        future_idx = i + lookback_window + forecast_horizon - 1

        X_list.append(X_scaled[i : i + lookback_window])
        y_list.append(prices[future_idx] / prices[today_idx] - 1)
        dates_list.append(dates[today_idx])

    X = np.array(X_list)
    y = np.array(y_list)
    dates = np.array(dates_list)

    print(f"X: {X.shape} | y: {y.shape} | dates: {dates.shape}")
    return X, y, dates

In [134]:
X_train_tensor, y_train_tensor, dates_train = make_windows(
    train_featured_scaled_stock_tensor,
    train_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)
X_val_tensor, y_val_tensor, dates_val = make_windows(
    val_featured_scaled_stock_tensor,
    val_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)
X_test_tensor, y_test_tensor, dates_test = make_windows(
    test_featured_scaled_stock_tensor,
    test_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)

X: (646, 20, 781) | y: (646,) | dates: (646,)
X: (69, 20, 781) | y: (69,) | dates: (69,)
X: (84, 20, 781) | y: (84,) | dates: (84,)


In [135]:
target_scaler = StandardScaler()
y_train_tensor = target_scaler.fit_transform(y_train_tensor.reshape(-1, 1)).flatten()
y_val_tensor = target_scaler.transform(y_val_tensor.reshape(-1, 1)).flatten()
y_test_tensor = target_scaler.transform(y_test_tensor.reshape(-1, 1)).flatten()

print(
    f"y_train_tensor — mean: {y_train_tensor.mean():.4f} | std: {y_train_tensor.std():.4f}"
)
print(
    f"y_val_tensor   — mean: {y_val_tensor.mean():.4f}   | std: {y_val_tensor.std():.4f}"
)
print(
    f"y_test_tensor  — mean: {y_test_tensor.mean():.4f}  | std: {y_test_tensor.std():.4f}"
)

y_train_tensor — mean: -0.0000 | std: 1.0000
y_val_tensor   — mean: -0.0559   | std: 0.5845
y_test_tensor  — mean: -0.0559  | std: 0.9010


In [136]:
type(X_train_tensor)

numpy.ndarray

In [137]:
dates_train[:1]

array(['2009-10-27T00:00:00.000000000'], dtype='datetime64[ns]')

## Validate windows

In [138]:
number_of_sample_windows = X_train_tensor.shape[0]
print(f"Number of sample windows: {number_of_sample_windows}")

Number of sample windows: 646


In [139]:
sample_idx = 0

if sample_idx < 0 or sample_idx > number_of_sample_windows - 1:
    raise ValueError(f"sample_idx must be between 0 and {number_of_sample_windows - 1}")

# ── raw index positions this sample corresponds to ──
today_idx = sample_idx + LOOKBACK_WINDOW - 1
future_idx = sample_idx + LOOKBACK_WINDOW + FORECAST_HORIZON - 1

# ── feature names output by the preprocessor ──
feature_names = (
    numerical
    + preprocessor.named_transformers_["nom"].get_feature_names_out(nominal).tolist()
    + ordinal
)

print("=" * 60)
print(f"SAMPLE INDEX: {sample_idx} / {number_of_sample_windows - 1}")
print("=" * 60)

print(f"\n── Input window dates ──")
print(f"  From : {train_featured_stock_df['date'].iloc[sample_idx]}")
print(f"  To   : {train_featured_stock_df['date'].iloc[today_idx]}  ← today")

print(f"\n── Target ──")
print(
    f"  Today  date               : {train_featured_stock_df['date'].iloc[today_idx]}"
)
print(
    f"  Future date               : {train_featured_stock_df['date'].iloc[future_idx]}"
)
print(
    f"  Today  price              : {train_featured_stock_df['adjust'].iloc[today_idx]}"
)
print(
    f"  Future price              : {train_featured_stock_df['adjust'].iloc[future_idx]}"
)
print(f"  y (standardized return)   : {y_train_tensor[sample_idx]:.6f}")

print(f"\n── X[0] — scaled input window — shape {X_train_tensor[sample_idx].shape} ──")
pd.DataFrame(
    X_train_tensor[sample_idx],
    columns=feature_names,
    index=train_featured_stock_df["date"].iloc[sample_idx : today_idx + 1].values,
)

SAMPLE INDEX: 0 / 645

── Input window dates ──
  From : 2009-09-30 00:00:00
  To   : 2009-10-27 00:00:00  ← today

── Target ──
  Today  date               : 2009-10-27 00:00:00
  Future date               : 2009-11-03 00:00:00
  Today  price              : 8.07
  Future price              : 7.84
  y (standardized return)   : -0.690533

── X[0] — scaled input window — shape (20, 781) ──


,close,change,matching_volume,matching_value,open,high,low,percent_change,number_of_buy_orders,buy_volume,...,upcom_index_date_is_month_end_false,upcom_index_date_is_month_end_true,upcom_index_date_is_quarter_start_false,upcom_index_date_is_quarter_start_true,upcom_index_date_is_quarter_end_false,upcom_index_date_is_quarter_end_true,upcom_index_date_is_year_start_false,upcom_index_date_is_year_end_false,upcom_index_date_is_year_end_true,ht_dcphase_quadrant
2009-09-30,0.085423,-0.494898,-0.287758,-0.291884,0.107699,0.100112,0.092561,-0.511162,-0.162002,-0.275802,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0
2009-10-01,0.043815,-0.961482,-0.085193,-0.155784,0.086875,0.058958,0.071467,-0.988169,-0.157694,-0.205965,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0
2009-10-02,-0.018596,-1.428067,0.586286,0.277123,0.003580,-0.023349,-0.034003,-1.490549,0.283898,0.315965,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,2.0
2009-10-05,-0.018596,-0.028313,-0.418884,-0.414591,0.003580,-0.023349,-0.034003,-0.039229,-0.163079,-0.354405,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,2.0
2009-10-06,-0.039400,-0.494898,-0.599353,-0.538471,0.003580,-0.023349,-0.012909,-0.536534,-0.156617,-0.401348,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-07,-0.018596,0.438271,-0.431037,-0.419948,-0.017243,-0.023349,-0.012909,0.463151,-0.293402,-0.187766,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-08,-0.018596,-0.028313,-0.614734,-0.548515,0.003580,-0.023349,-0.012909,-0.039229,-0.317098,-0.413456,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-09,0.002208,0.438271,-0.184899,-0.246350,0.003580,-0.002772,0.008185,0.458077,-0.307404,-0.135122,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-12,0.064619,1.371440,-0.229265,-0.266104,0.024404,0.038381,0.029279,1.437464,-0.561589,-0.170964,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-13,0.023011,-0.961482,-0.534033,-0.481386,0.066052,0.038381,0.008185,-0.998318,-0.485118,-0.318104,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0


## Create metadata JSON

In [140]:
metadata = {
    "stock_code": STOCK_CODE,
    "lookback_window": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "stride": STRIDE,
    "train_range": [
        train_featured_stock_df.date.min().strftime("%Y-%m-%d"),
        train_featured_stock_df.date.max().strftime("%Y-%m-%d"),
    ],
    "train_shape": X_train_tensor.shape,
    "val_range": [
        val_featured_stock_df.date.min().strftime("%Y-%m-%d"),
        val_featured_stock_df.date.max().strftime("%Y-%m-%d"),
    ],
    "val_shape": X_val_tensor.shape,
    "test_range": [
        test_featured_stock_df.date.min().strftime("%Y-%m-%d"),
        test_featured_stock_df.date.max().strftime("%Y-%m-%d"),
    ],
    "test_shape": X_test_tensor.shape,
}

metadata

{'stock_code': 'vcb',
 'lookback_window': 20,
 'forecast_horizon': 5,
 'stride': 5,
 'train_range': ['2009-09-30', '2022-12-30'],
 'train_shape': (646, 20, 781),
 'val_range': ['2023-01-03', '2024-06-28'],
 'val_shape': (69, 20, 781),
 'test_range': ['2024-07-01', '2026-04-29'],
 'test_shape': (84, 20, 781)}

## Write to folder

In [141]:
TRAIN_TEST_SET_DIR

'../../train_test_set'

In [142]:
TRAIN_TEST_SET_STOCK_CODE = (
    f"{TRAIN_TEST_SET_DIR}/{STOCK_CODE}_{LOOKBACK_WINDOW}_{FORECAST_HORIZON}_{STRIDE}"
)
os.makedirs(TRAIN_TEST_SET_STOCK_CODE, exist_ok=True)
TRAIN_TEST_SET_STOCK_CODE

'../../train_test_set/vcb_20_5_5'

In [143]:
metadata_path = f"{TRAIN_TEST_SET_STOCK_CODE}/metadata.json"
print(metadata_path)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
    print(f"Metadata written to {metadata_path}")

../../train_test_set/vcb_20_5_5/metadata.json
Metadata written to ../../train_test_set/vcb_20_5_5/metadata.json


In [144]:
import joblib

train_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/train_featured_stock_df.csv", index=False
)
val_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/val_featured_stock_df.csv", index=False
)
test_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/test_featured_stock_df.csv", index=False
)

np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/train_featured_scaled_stock_tensor.npy",
    train_featured_scaled_stock_tensor,
)
np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/val_featured_scaled_stock_tensor.npy",
    val_featured_scaled_stock_tensor,
)
np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/test_featured_scaled_stock_tensor.npy",
    test_featured_scaled_stock_tensor,
)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_train_tensor.npy", X_train_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_val_tensor.npy", X_val_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_test_tensor.npy", X_test_tensor)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_train_tensor.npy", y_train_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_val_tensor.npy", y_val_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_test_tensor.npy", y_test_tensor)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_train.npy", dates_train)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_val.npy", dates_val)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_test.npy", dates_test)

joblib.dump(target_scaler, f"{TRAIN_TEST_SET_STOCK_CODE}/target_scaler.pkl")

['../../train_test_set/vcb_20_5_5/target_scaler.pkl']